<div align="center">

# Day 04: Reinforcement Learning for Language Models

**Reinforcement Learning and Language Model Alignment (VTR UGE 21), hands-on notebook**

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

[Day overview](https://github.com/utkukose/rl-alignment-veltech-NB-lecture/tree/main/days/day-04) · [Lecture page](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-04/lecture.html) · [Interactive lab](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-04/lab.html) · [PDF notes](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-04/Day04_Lecture_Notes.pdf)

</div>

## How to use this notebook

This notebook is the hands-on part of the day. The concepts are explained on the [lecture page](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-04/lecture.html), and each section that applies a part of the lecture starts with a link to it. Save your own copy first with File, Save a copy in Drive, then run the cells in order with Shift+Enter. Exercises check your answer as soon as you run them. Each exercise has a reference solution in a collapsed cell: Try first, then open it. Running all cells at once also runs the reference solutions. Nothing needs to be installed.

## 0. Setup

The first cell prepares the environment and defines the helper `check`, which compares an answer with the expected value and prints immediate feedback.

In [ ]:
# Environment: Everything used by this notebook is preinstalled in Google Colab.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from dataclasses import dataclass, field, asdict
from itertools import product
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
plt.rcParams.update({'figure.dpi': 110, 'figure.figsize': (7, 3.4), 'axes.grid': True, 'grid.alpha': 0.3, 'font.size': 9})

def check(name, value, expected, tol=1e-6):
    """Compares an answer with the expected value and prints immediate feedback."""
    if value is None:
        print(f"{name}: not answered yet. Replace None with your answer and run the cell again.")
        return False
    try:
        ok = abs(float(value) - float(expected)) <= tol * max(1.0, abs(float(expected)))
    except (TypeError, ValueError):
        ok = value == expected
    print(f"{name}: " + ("correct." if ok else "not yet. Compare your steps with the lecture and try again."))
    return ok


import time
print('ready')

## Python step 4: Text as integers

A language model never sees letters or words: It sees integers. Each integer stands for a token, as one unit of text is called, here a word. This step turns a sentence into a sequence of integers and back, pads it to a fixed length, samples a next token from a probability vector and computes the log-probability of a sequence &#91;[7](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-04/lecture.html#ref-7)&#93;.

### Words and a vocabulary

In [ ]:
sentence = "the cat slept on the mat"
words = sentence.split()
vocab = ["<pad>", "<bos>", "<eos>"] + sorted(set(words))
stoi = {w: i for i, w in enumerate(vocab)}
print(words)
print(stoi)

`split` cuts a string at spaces into a list of words. `set` removes duplicates, `sorted` orders them, and the three special tokens mark padding, the beginning and the end of a sentence. The dictionary `stoi`, string to integer, gives every token its number.

### Encoding, decoding and padding

In [ ]:
ids = [stoi["<bos>"]] + [stoi[w] for w in words] + [stoi["<eos>"]]
padded = ids + [stoi["<pad>"]] * (10 - len(ids))
print(ids)
print(padded)
print(" ".join(vocab[i] for i in ids[1:-1]))

A list comprehension encodes every word. Padding brings every sequence to the same length, so that many sentences fit into one array. Decoding looks each integer up in the vocabulary list.

### Sampling the next token

In [ ]:
import numpy as np
p = np.array([0.0, 0.0, 0.1, 0.05, 0.6, 0.25])      # probabilities of six tokens
rng = np.random.default_rng(0)
draws = [int((p.cumsum() > rng.random()).argmax()) for _ in range(1000)]
print(np.bincount(draws, minlength=6) / 1000)

`cumsum` stacks the probabilities into the intervals of a line from 0 to 1; a uniform random number falls into the interval of token $k$ with probability $p_k$, and `argmax` finds that interval. A thousand draws reproduce the probabilities closely. The language model samples every token of a sentence this way.

### The log-probability of a sequence

In [ ]:
probs = [0.9, 0.5, 0.8, 0.95]          # model probabilities of the four chosen tokens
logp = np.log(probs).sum()
print("log-probability:", round(float(logp), 3), "  probability:", round(float(np.exp(logp)), 3))

The probability of a sequence is the product of the probabilities of its tokens, and its logarithm is a sum, which is numerically safe for long sequences. Here $\log 0.9 + \log 0.5 + \log 0.8 + \log 0.95 = -1.073$, with the natural logarithm `np.log`. The lecture writes this sum as $\log \pi_\theta(y)$, the log-probability of the sentence $y$ under the model $\pi_\theta$. REINFORCE, the policy gradient algorithm of Day 3, and direct preference optimisation (DPO), the method of Day 5, both work with this sum.

**Quick check.** Why do language-model computations use log-probabilities instead of probabilities?

<details><summary>Answer</summary>

Because the probability of a sequence is a product of many numbers below one. Such a product quickly becomes too small for the computer to represent, which is called an underflow, while the sum of the logarithms stays in a safe range.

</details>

### Exercises for Python step 4

**Exercise 4.1.** How many distinct words does the sentence contain? Store the number in `n_distinct`.

In [ ]:
n_distinct = None   # your answer

check("Exercise 4.1", n_distinct, 5)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
n_distinct = len(set(words))

check("Exercise 4.1", n_distinct, 5)

**Exercise 4.2.** Decode the integers `[7, 3, 6]` with `vocab` and store the resulting string in `decoded`.

In [ ]:
decoded = None   # your answer

check("Exercise 4.2", decoded, "the cat slept")

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
decoded = " ".join(vocab[i] for i in [7, 3, 6])

check("Exercise 4.2", decoded, "the cat slept")

**Exercise 4.3.** Store the log-probability of a sequence of three tokens, each with probability 0.5, rounded to three decimals, in `lp3`.

In [ ]:
lp3 = None   # your answer

check("Exercise 4.3", lp3, -2.079)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
lp3 = round(float(np.log([0.5, 0.5, 0.5]).sum()), 3)

check("Exercise 4.3", lp3, -2.079)

## 1. The large TokenWorld and its gold reward

*Lecture: [A small language model](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-04/lecture.html#a-small-language-model)*

Sections 1 to 3 build a small language model and fine-tune it. Sections 4 and 5 learn a reward model from preference pairs. Sections 6 and 7 improve the language model by reinforcement learning from human feedback (RLHF) with a penalty on the Kullback-Leibler (KL) divergence.

**From the formulas to the code.** `make_corpus` builds each sentence from one entry of `SUBJ`, `VERB` and `PLACE` and, when `rng.random() < 0.45`, one entry of `EXTRA`. `gold_reward` is the formula $r_{\mathrm{gold}}(y)$ of the lecture, one line per rule: `len(CALM.intersection(w))` is the number of different calm words, so a repeated calm word counts once, and `len(set(content)) < len(content)` detects a repeated content word. `encode` turns a sentence into 12 integers: `<bos>`, the words, `<eos>` and then `<pad>` up to `MAXLEN`. `GRAMMAR` is the set of all 1512 sentences that the four lists allow, and the later sections test whether a sentence belongs to it.

In [ ]:
# The large TokenWorld: short English sentences built from four slots, and the gold reward.
SUBJ = ["the cat", "the dog", "a bird", "my friend", "the child", "a robot"]
VERB = ["sat", "ran", "slept", "sang", "waited", "smiled"]
PLACE = ["on the mat", "in the garden", "by the window", "near the door", "under the tree", "beside the lamp"]
EXTRA = ["quietly", "happily", "again", "for a while", "today", "at dawn"]

def make_corpus(n=4000, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed); out = []
    for _ in range(n):
        parts = [rng.choice(SUBJ), rng.choice(VERB), rng.choice(PLACE)]
        if rng.random() < 0.45:
            parts.append(rng.choice(EXTRA))
        out.append(" ".join(parts))
    return out

CALM = {"quietly", "slept", "waited", "beside", "under", "garden", "window", "dawn"}
LOUD = {"sang", "ran", "happily", "again", "robot"}
FUNC = {"the", "a", "my", "in", "on", "by", "at", "near", "under", "beside", "for", "while"}

def gold_reward(text):
    """The reward that is wanted: a calm, well-formed, non-repetitive, reasonably short sentence."""
    w = text.split()
    if not w:
        return -2.0
    content = [x for x in w if x not in FUNC]
    r = 1.0 if w[0] in ("the", "a", "my") else -0.5          # a grammatical opening
    r += 0.45 * len(CALM.intersection(w)) - 0.45 * len(LOUD.intersection(w))
    r -= 0.10 * max(0, len(w) - 6)                           # a mild preference for brevity
    if len(set(content)) < len(content):
        r -= 0.8                                             # repeated content words
    return float(r)

corpus = make_corpus()
VOCAB = ["<pad>", "<bos>", "<eos>"] + sorted({w for s in corpus for w in s.split()})
STOI = {w: i for i, w in enumerate(VOCAB)}; PAD, BOS, EOS = 0, 1, 2; V = len(VOCAB); MAXLEN = 12

def encode(s):
    ids = [BOS] + [STOI[w] for w in s.split()][:MAXLEN - 2] + [EOS]
    return ids + [PAD] * (MAXLEN - len(ids))

def decode(row):
    out = []
    for i in row[1:]:
        if i == EOS or i == PAD:
            break
        out.append(VOCAB[i])
    return " ".join(out)

IDS = np.array([encode(s) for s in corpus])
GRAMMAR = {" ".join([s, v, p] + ([e] if e else [])) for s in SUBJ for v in VERB for p in PLACE for e in EXTRA + [None]}
print(f"{len(corpus)} sentences, vocabulary of {V} tokens; example: '{corpus[0]}'")
print(f"gold reward of the corpus: mean {np.mean([gold_reward(s) for s in corpus]):.3f}")

## 2. A neural language model in NumPy &#91;[4](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-04/lecture.html#ref-4)&#93;

*Lecture: [A small language model](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-04/lecture.html#a-small-language-model)*

**Python move: one backward pass for three objectives.** A backward pass computes a gradient, from the output of a model back to its parameters. `grad_weighted(ids, w)` returns the gradient of the negative log-probabilities of the sequences, each weighted by `w`; the weights decide whether the update is fine-tuning, REINFORCE or DPO.

**From the formulas to the code.** `forward` computes the four lines of the model: `ctx` holds the three previous tokens of every position, `X` is the input $x_t$, `Hh` the hidden layer $h_t$, `logits` the scores $z_t$ and `lsm` the logarithm of the softmax, $\log \pi_\theta(v \mid y_1, \ldots, y_{t-1})$ for every token $v$. The code counts positions from 0 and sets the score of `<pad>` to $-10^9$, so that padding is never written. `seq_logprob` adds the log-probabilities of the tokens of each sentence, with a mask that leaves the padding out: $\log \pi_\theta(y)$. `grad_weighted(ids, w)` is the backward pass, the gradient of $L_w = -\sum_n w_n \log \pi_\theta(y^{(n)})$ with respect to every parameter. `adam_step` clips this gradient to length 1 and applies the Adam update of the lecture, and `sample` draws each token with the cumulative sum of the Python step and ends every sentence after ten words.

In [ ]:
#@title A neural language model written with NumPy (run this cell; open it to read the code)
class NeuralLM:
    """A neural language model: each next token is predicted from the previous K tokens and the position.

    Every training objective of Days 4 and 5 is a weighted sum of token log-probabilities, so one backward pass,
    `grad_weighted`, serves supervised fine-tuning, REINFORCE and DPO alike."""
    def __init__(self, K=3, d=16, H=128, seed=0):
        rng = np.random.default_rng(seed); self.K, self.d = K, d
        self.P = {"E": rng.normal(0, 0.3, (V, d)), "pos": rng.normal(0, 0.3, (MAXLEN, d)),
                  "W1": rng.normal(0, 1 / np.sqrt((K + 1) * d), ((K + 1) * d, H)), "b1": np.zeros(H),
                  "W2": rng.normal(0, 1 / np.sqrt(H), (H, V)), "b2": np.zeros(V)}
        self.m = {k: np.zeros_like(v) for k, v in self.P.items()}; self.v = {k: np.zeros_like(v) for k, v in self.P.items()}
        self.t = 0

    def copy(self):
        c = NeuralLM.__new__(NeuralLM); c.K, c.d = self.K, self.d
        c.P = {k: v.copy() for k, v in self.P.items()}
        c.m = {k: np.zeros_like(v) for k, v in self.P.items()}; c.v = {k: np.zeros_like(v) for k, v in self.P.items()}; c.t = 0
        return c

    def _context(self, ids):
        N, T = ids.shape
        ext = np.concatenate([np.full((N, self.K - 1), BOS), ids[:, :-1]], axis=1)
        return np.stack([ext[:, k:k + T - 1] for k in range(self.K)], axis=2)       # (N, T-1, K)

    def forward(self, ids):
        """Log-probabilities of every next token, shape (N, MAXLEN-1, V)."""
        N, T = ids.shape; ctx = self._context(ids); pos = np.broadcast_to(self.P["pos"][:T - 1], (N, T - 1, self.d))
        X = np.concatenate([self.P["E"][ctx].reshape(N, T - 1, -1), pos], axis=2)
        Hh = np.tanh(X @ self.P["W1"] + self.P["b1"]); logits = Hh @ self.P["W2"] + self.P["b2"]
        logits[..., PAD] = -1e9                                                         # padding is never generated
        lsm = logits - logits.max(-1, keepdims=True); lsm = lsm - np.log(np.exp(lsm).sum(-1, keepdims=True))
        self.cache = (ids, ctx, X, Hh, lsm); return lsm

    def token_logprobs(self, ids):
        lsm = self.forward(ids); y = ids[:, 1:]
        return np.take_along_axis(lsm, y[..., None], 2)[..., 0], (y != PAD).astype(float)

    def seq_logprob(self, ids):
        lp, mask = self.token_logprobs(ids); return (lp * mask).sum(1)

    def grad_weighted(self, ids, w):
        """Gradient of  -sum_n w_n * log pi(sequence n)  with respect to every parameter."""
        lsm = self.forward(ids); ids_, ctx, X, Hh, _ = self.cache; N, T1 = ctx.shape[:2]; y = ids[:, 1:]
        Wt = (y != PAD) * np.asarray(w, float)[:, None]
        G = np.exp(lsm) * Wt[..., None]; np.put_along_axis(G, y[..., None], np.take_along_axis(G, y[..., None], 2) - Wt[..., None], 2)
        g = {"W2": Hh.reshape(-1, Hh.shape[-1]).T @ G.reshape(-1, V), "b2": G.sum((0, 1))}
        dpre = (G @ self.P["W2"].T) * (1 - Hh ** 2)
        g["W1"] = X.reshape(-1, X.shape[-1]).T @ dpre.reshape(-1, dpre.shape[-1]); g["b1"] = dpre.sum((0, 1))
        dX = dpre @ self.P["W1"].T
        g["E"] = np.zeros_like(self.P["E"]); np.add.at(g["E"], ctx, dX[..., :self.K * self.d].reshape(N, T1, self.K, self.d))
        g["pos"] = np.zeros_like(self.P["pos"]); g["pos"][:T1] = dX[..., self.K * self.d:].sum(0)
        return g

    def adam_step(self, g, lr=3e-3, clip=1.0):
        norm = np.sqrt(sum((v ** 2).sum() for v in g.values())); scale = min(1.0, clip / (norm + 1e-12)); self.t += 1
        for k in self.P:
            gk = g[k] * scale; self.m[k] = 0.9 * self.m[k] + 0.1 * gk; self.v[k] = 0.999 * self.v[k] + 0.001 * gk ** 2
            self.P[k] -= lr * (self.m[k] / (1 - 0.9 ** self.t)) / (np.sqrt(self.v[k] / (1 - 0.999 ** self.t)) + 1e-8)

    def sample(self, n, seed=0, temperature=1.0):
        """Generate n sequences token by token."""
        rng = np.random.default_rng(seed); ids = np.full((n, MAXLEN), PAD); ids[:, 0] = BOS; done = np.zeros(n, bool)
        for t in range(1, MAXLEN):
            lsm = self.forward(ids[:, :t + 1])[:, t - 1] / temperature
            p = np.exp(lsm - lsm.max(1, keepdims=True)); p /= p.sum(1, keepdims=True)
            nxt = (p.cumsum(1) > rng.random((n, 1))).argmax(1)
            if t == MAXLEN - 1:
                nxt = np.full(n, EOS)                                                # every sequence ends by the cap
            nxt[done] = PAD; ids[:, t] = nxt; done |= nxt == EOS
            if done.all():
                break
        return ids

## 3. Supervised fine-tuning

*Lecture: [Supervised fine-tuning](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-04/lecture.html#supervised-fine-tuning)*

**From the formulas to the code.** SFT is short for supervised fine-tuning. In `train_sft`, the weights `w = np.ones(len(b)) / ((batch[:, 1:] != PAD).sum())` are $1 / M$, one over the number of tokens of the batch, so `lm.grad_weighted(batch, w)` is the gradient of the loss $L_{\mathrm{SFT}}$ and `lm.adam_step` takes one step against it. `np.array_split` cuts the shuffled corpus into 31 batches per epoch. The printed mean token log-probability is $-L_{\mathrm{SFT}}$, measured on the first 500 sentences of the corpus, and `t in GRAMMAR` tests whether a sampled sentence follows the grammar.

In [ ]:
import time
def train_sft(epochs=12, lr=3e-3, seed=RANDOM_STATE):
    """Supervised fine-tuning: maximise the log-probability of the corpus, token by token."""
    lm = NeuralLM(seed=seed); rng = np.random.default_rng(seed)
    for ep in range(epochs):
        for b in np.array_split(rng.permutation(len(IDS)), len(IDS) // 128):
            batch = IDS[b]; w = np.ones(len(b)) / ((batch[:, 1:] != PAD).sum())
            lm.adam_step(lm.grad_weighted(batch, w), lr=lr)
    return lm

t0 = time.time(); sft = train_sft()
lp, mask = sft.token_logprobs(IDS[:500])
print(f"SFT trained in {time.time() - t0:.1f} s; mean token log-probability on the corpus {(lp * mask).sum() / mask.sum():.3f}")
o_sft = sft.sample(512, seed=1); t_sft = [decode(r) for r in o_sft]
g_sft = np.array([gold_reward(t) for t in t_sft])
print(f"SFT policy, gold reward over 512 samples: mean {g_sft.mean():.3f}, standard deviation {g_sft.std():.3f}")
print(f"share of samples that follow the grammar of the corpus: {np.mean([t in GRAMMAR for t in t_sft]):.3f}")
for t in t_sft[:5]:
    print(f"   {gold_reward(t):+.2f}  {t}")

## 4. Preference pairs from a noisy annotator

*Lecture: [Preferences and the annotator](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-04/lecture.html#preferences-and-the-annotator)*

**From the formulas to the code.** `make_pairs` samples `2 * n_pairs` sentences and compares the gold rewards `ga` and `gb` of each pair. A tie is skipped with `continue`. The test `rng.random() < noise` is true with probability $\eta$, the error rate of the annotator, and then winner and loser change places. The arrays `W` and `L` hold the sentences that the annotator named as winners $y_w$ and losers $y_l$, flipped labels included, and `flipped` counts the errors. The last two lines measure the difference of the gold rewards on a second sample of 500 pairs, ties included.

In [ ]:
def make_pairs(model, n_pairs=1500, noise=0.10, seed=7):
    """Sample two completions and ask a noisy annotator, who follows the gold reward, which one is better."""
    rng = np.random.default_rng(seed); o = model.sample(2 * n_pairs, seed=seed); txt = [decode(r) for r in o]
    W, L, flipped = [], [], 0
    for i in range(n_pairs):
        ga, gb = gold_reward(txt[2 * i]), gold_reward(txt[2 * i + 1])
        if abs(ga - gb) < 1e-9:
            continue                                        # a tie: an annotator would skip it
        win, lose = (o[2 * i], o[2 * i + 1]) if ga > gb else (o[2 * i + 1], o[2 * i])
        if rng.random() < noise:
            win, lose = lose, win; flipped += 1             # an annotation error
        W.append(win); L.append(lose)
    return np.array(W), np.array(L), flipped

Wp, Lp, flipped = make_pairs(sft, noise=0.10)
print(f"{len(Wp)} usable pairs, {flipped} labels flipped by annotator error ({flipped / len(Wp):.1%})")
gv = np.array([gold_reward(decode(r)) for r in sft.sample(1000, seed=11)]); diffs = np.abs(gv[0::2] - gv[1::2])
print(f"median gold difference within a pair: {np.median(diffs):.3f}; pairs closer than 0.2: {(diffs < 0.2).mean():.1%}")

## 5. A Bradley-Terry reward model &#91;[1](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-04/lecture.html#ref-1), [8](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-04/lecture.html#ref-8)&#93;

*Lecture: [Learning a reward from comparisons](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-04/lecture.html#learning-a-reward-from-comparisons)*

**From the formulas to the code.** In `RewardModel.score`, `tok = ids > EOS` marks the words of a sentence, `n` is their number $|y|$, kept at 1 or more to avoid a division by zero, and `F` holds the features $F(y)$: the mean embedding and `n / 10.0`. The returned value is the score $r_\phi(y)$. In `train_rm`, `sw - sl` is the difference $\Delta_i$ and `q = 1 / (1 + np.exp(sw - sl))` is $1 - \sigma(\Delta_i)$. The calls `rm.grad(-q / len(b))` and `rm.grad(q / len(b))` pass the slopes of the mean loss with respect to the winner and loser scores back to the parameters. Each of them follows a call of `rm.score`, because `grad` uses the values that the last call of `score` stored. `adam` is the Adam update without clipping, `acc` is the held-out accuracy on the last 15 percent of the pairs, and `np.corrcoef` computes the correlation coefficient $\rho$.

In [ ]:
def adam(P, g, state, lr):
    state["t"] = state.get("t", 0) + 1; t = state["t"]
    for k in P:
        m = state.setdefault("m" + k, np.zeros_like(P[k])); v = state.setdefault("v" + k, np.zeros_like(P[k]))
        m[...] = 0.9 * m + 0.1 * g[k]; v[...] = 0.999 * v + 0.001 * g[k] ** 2
        P[k] -= lr * (m / (1 - 0.9 ** t)) / (np.sqrt(v / (1 - 0.999 ** t)) + 1e-8)

class RewardModel:
    """The mean of the token embeddings and the length feed a small network that outputs one score.
    Like every learned reward, it is an approximation: It sees which words occur, not their order or repetition."""
    def __init__(self, d=16, H=32, seed=0):
        rng = np.random.default_rng(seed)
        self.P = {"E": rng.normal(0, 0.3, (V, d)), "U1": rng.normal(0, 1 / np.sqrt(d + 1), (d + 1, H)), "c1": np.zeros(H),
                  "u2": rng.normal(0, 1 / np.sqrt(H), H), "c2": np.zeros(1)}
        self.state = {}
    def score(self, ids):
        tok = ids > EOS; n = tok.sum(1, keepdims=True).clip(1)
        F = np.concatenate([(self.P["E"][ids] * tok[..., None]).sum(1) / n, n / 10.0], 1)
        Hh = np.tanh(F @ self.P["U1"] + self.P["c1"]); self.cache = (ids, F, tok, n, Hh)
        return Hh @ self.P["u2"] + self.P["c2"][0]
    def grad(self, gs):
        ids, F, tok, n, Hh = self.cache
        g = {"u2": Hh.T @ gs, "c2": np.array([gs.sum()])}; dpre = np.outer(gs, self.P["u2"]) * (1 - Hh ** 2)
        g["U1"] = F.T @ dpre; g["c1"] = dpre.sum(0); dF = dpre @ self.P["U1"].T
        g["E"] = np.zeros_like(self.P["E"]); np.add.at(g["E"], ids, dF[:, None, :-1] * tok[..., None] / n[..., None])
        return g

def train_rm(W, L, epochs=30, lr=3e-3, seed=0, holdout=0.15):
    """Bradley-Terry: maximise log sigmoid(r(winner) - r(loser)) over the training pairs."""
    rm = RewardModel(seed=seed); rng = np.random.default_rng(seed); ntr = int((1 - holdout) * len(W))
    for ep in range(epochs):
        for b in np.array_split(rng.permutation(ntr), max(1, ntr // 128)):
            sw = rm.score(W[b]); sl = rm.score(L[b]); q = 1 / (1 + np.exp(sw - sl))     # q = 1 - sigmoid(sw - sl)
            rm.score(W[b]); gw = rm.grad(-q / len(b)); rm.score(L[b]); gl = rm.grad(q / len(b))
            adam(rm.P, {k: gw[k] + gl[k] for k in gw}, rm.state, lr)
    acc = float((rm.score(W[ntr:]) > rm.score(L[ntr:])).mean())
    return rm, acc

rm, acc = train_rm(Wp, Lp)
rm_on_sft = rm.score(o_sft); rho = float(np.corrcoef(rm_on_sft, g_sft)[0, 1])
print(f"reward model: held-out pairwise accuracy {acc:.3f}; correlation with the gold reward on SFT samples {rho:.3f}")

**Exercise A.** Store the held-out pairwise accuracy of the reward model in `rm_acc`.

In [ ]:
rm_acc = None   # your answer

check("Exercise A", rm_acc, acc)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
rm_acc = acc

check("Exercise A", rm_acc, acc)

In [ ]:
rows = []
for nz in (0.0, 0.05, 0.10, 0.20, 0.35):
    Wn, Ln, _ = make_pairs(sft, noise=nz, seed=7); r_, a_ = train_rm(Wn, Ln, seed=0)
    rows.append(dict(annotator_noise=nz, pairs=len(Wn), heldout_accuracy=a_,
                     correlation_with_gold=float(np.corrcoef(r_.score(o_sft), g_sft)[0, 1])))
noise_tab = pd.DataFrame(rows)
print(noise_tab.round(3).to_string(index=False))

## 6. RLHF with a KL penalty &#91;[2](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-04/lecture.html#ref-2), [3](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-04/lecture.html#ref-3)&#93;

*Lecture: [RLHF with a KL penalty](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-04/lecture.html#rlhf-with-a-kl-penalty)*

**From the formulas to the code.** The function `rlhf` is the third stage, reinforcement learning from human feedback: `pol` is the policy $\pi_\theta$ and `ref` the reference policy $\pi_{\mathrm{ref}}$. The line `kl = pol.seq_logprob(seqs) - ref.seq_logprob(seqs)` computes the log-ratio of every sampled sentence, `reward = rm.score(seqs) - beta * kl` is the penalised reward $\tilde{r}(y)$ and `adv` is the advantage $A^{(n)}$. `pol.grad_weighted(seqs, adv / batch)` returns the gradient of $L_w$ with $w_n = A^{(n)} / B$, which is $-\hat{g}$, so the Adam step against it moves the parameters along $\hat{g}$. The column `kl` of `hist` is the mean log-ratio of 192 fresh sentences, the estimate of the KL divergence. The row of iteration 0 is recorded after the first update.

In [ ]:
def rlhf(sft, rm, beta=0.2, iters=300, batch=64, lr=1e-3, seed=0, track=25, eval_n=192):
    """REINFORCE with the KL penalty folded into the reward:  r(y) = RM(y) - beta * [log pi(y) - log pi_ref(y)]."""
    pol, ref = sft.copy(), sft; hist = []
    for it in range(iters):
        seqs = pol.sample(batch, seed=1000 * seed + it)
        kl = pol.seq_logprob(seqs) - ref.seq_logprob(seqs)                # a one-sample estimate of the KL
        reward = rm.score(seqs) - beta * kl
        adv = (reward - reward.mean()) / (reward.std() + 1e-6)
        pol.adam_step(pol.grad_weighted(seqs, adv / batch), lr=lr)
        if it % track == 0 or it == iters - 1:
            ev = pol.sample(eval_n, seed=50000 + it); tt = [decode(r) for r in ev]
            hist.append(dict(iter=it, gold=np.mean([gold_reward(t) for t in tt]), reward_model=float(rm.score(ev).mean()),
                             kl=float((pol.seq_logprob(ev) - ref.seq_logprob(ev)).mean()),
                             length=np.mean([len(t.split()) for t in tt]), in_grammar=np.mean([t in GRAMMAR for t in tt])))
    return pol, pd.DataFrame(hist)

In [ ]:
t0 = time.time(); pol, hist = rlhf(sft, rm, beta=0.2, iters=300, seed=0)
print(f"RLHF with beta = 0.2 finished in {time.time() - t0:.1f} s\n")
print(hist.round(3).to_string(index=False))
print("\nsamples:", *[decode(r) for r in pol.sample(4, seed=3)], sep="\n   ")

## 7. The KL weight as a dial, and reward hacking &#91;[5](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-04/lecture.html#ref-5), [6](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-04/lecture.html#ref-6)&#93;

*Lecture: [Without the leash: reward hacking](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-04/lecture.html#without-the-leash-reward-hacking)*

**From the formulas to the code.** The loop calls `rlhf` once for each of seven values of `beta`, the KL weight $\beta$. For each final policy, `gold` and `in_grammar` are computed on 400 fresh sentences, and `distinct_outputs` is the number of different sentences among them. `peak_gold` is the highest gold reward of the 13 recorded evaluations, and `reward_model` and `kl` come from the last one. The second cell draws the figure of the lecture from the recorded evaluations of four runs.

In [ ]:
sweep, runs = [], {}
for beta in (0.0, 0.02, 0.05, 0.1, 0.2, 0.5, 1.0):
    p_, h_ = rlhf(sft, rm, beta=beta, iters=300, seed=0); runs[beta] = (p_, h_)
    texts = [decode(r) for r in p_.sample(400, seed=77)]
    sweep.append(dict(beta=beta, gold=np.mean([gold_reward(t) for t in texts]), peak_gold=h_.gold.max(),
                      reward_model=h_.reward_model.iloc[-1], kl=h_.kl.iloc[-1], distinct_outputs=len(set(texts)),
                      in_grammar=np.mean([t in GRAMMAR for t in texts])))
sweep = pd.DataFrame(sweep)
print(f"SFT baseline: gold {g_sft.mean():.3f}\n")
print(sweep.round(3).to_string(index=False))
h0 = runs[0.0][1]
print(f"\nwithout the penalty: gold peaks at {h0.gold.max():.2f} (iteration {int(h0.loc[h0.gold.idxmax(), 'iter'])}) "
      f"and ends at {h0.gold.iloc[-1]:.2f}, while the reward model score rises to {h0.reward_model.iloc[-1]:.2f}")
print("final samples without the penalty:", *sorted(set(decode(r) for r in runs[0.0][0].sample(50, seed=5)))[:3], sep="\n   ")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13, 3.3))
for beta, col in ((0.0, "#c0392b"), (0.05, "#e67e22"), (0.2, "#2980b9"), (1.0, "#27ae60")):
    h_ = runs[beta][1]
    ax[0].plot(h_["iter"], h_.gold, color=col, label=f"beta {beta}"); ax[1].plot(h_["iter"], h_.reward_model, color=col)
    ax[2].plot(h_.kl, h_.gold, "o-", ms=3, color=col)
ax[0].axhline(g_sft.mean(), ls="--", lw=1, color="#7f8c8d"); ax[0].set_title("gold reward: what is wanted", fontsize=9)
ax[1].set_title("reward-model score: what is optimised", fontsize=9); ax[2].set_title("gold reward against the KL", fontsize=9)
ax[0].set_xlabel("iteration"); ax[1].set_xlabel("iteration"); ax[2].set_xlabel("KL from the reference"); ax[0].legend(fontsize=8)
plt.tight_layout(); plt.show()

**Exercise B.** Which value of `beta` gives the highest final gold reward in the table `sweep`? Store it in `best_beta`.

In [ ]:
best_beta = None   # your answer

check("Exercise B", best_beta, float(sweep.loc[sweep.gold.idxmax(), "beta"]))

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
best_beta = float(sweep.loc[sweep.gold.idxmax(), "beta"])

check("Exercise B", best_beta, float(sweep.loc[sweep.gold.idxmax(), "beta"]))

**Exercise C.** How many distinct outputs among 400 samples does the policy trained without a penalty produce? Store the number in `n_hack`.

In [ ]:
n_hack = None   # your answer

check("Exercise C", n_hack, int(sweep.loc[sweep.beta == 0.0, "distinct_outputs"].iloc[0]))

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
n_hack = int(sweep.loc[sweep.beta == 0.0, "distinct_outputs"].iloc[0])

check("Exercise C", n_hack, int(sweep.loc[sweep.beta == 0.0, "distinct_outputs"].iloc[0]))

## 8. Your field: an annotator with another taste

*Lecture: [Your field](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-04/lecture.html#your-field)*

Change `TASTE` to `brief`, `calm` or `lively` and run the cell again.

**From the formulas to the code.** `pref` is the taste of the annotator as a function of the sentence: the gold reward, minus the number of words, the number of different calm words or the number of different loud words. `taste_pairs` labels each pair by the higher value of `pref`, skips ties and makes no errors. `train_rm` is the Bradley-Terry training of section 5, and `rho_taste` is the correlation coefficient $\rho$ between the scores of the new reward model and the gold reward on the 512 samples of the fine-tuned model.

In [ ]:
TASTE = "gold"      # choose: gold, brief, calm or lively

def taste_pairs(model, taste, n_pairs=1500, seed=21):
    """Pairs judged by an annotator with a given taste; ties are skipped."""
    pref = {"gold": gold_reward, "brief": lambda t: -len(t.split()),
            "calm": lambda t: len(CALM.intersection(t.split())), "lively": lambda t: len(LOUD.intersection(t.split()))}[taste]
    o = model.sample(2 * n_pairs, seed=seed); txt = [decode(r) for r in o]; W, L = [], []
    for i in range(n_pairs):
        a, b = pref(txt[2 * i]), pref(txt[2 * i + 1])
        if a != b:
            W.append(o[2 * i] if a > b else o[2 * i + 1]); L.append(o[2 * i + 1] if a > b else o[2 * i])
    return np.array(W), np.array(L)

W_taste, L_taste = taste_pairs(sft, TASTE)
rm_taste, acc_taste = train_rm(W_taste, L_taste)
score_taste = rm_taste.score(o_sft); rho_taste = float(np.corrcoef(score_taste, g_sft)[0, 1])
print(f"annotator taste '{TASTE}': {len(W_taste)} usable pairs")
print(f"   the reward model agrees with this annotator on {acc_taste:.1%} of held-out pairs")
print(f"   correlation of its scores with the gold reward: {rho_taste:+.3f}")
plt.figure(figsize=(5.5, 3.6)); plt.scatter(g_sft, score_taste, s=6, alpha=.4)
plt.xlabel("gold reward"); plt.ylabel(f"reward model trained on '{TASTE}'"); plt.title("what the reward model learned", fontsize=9); plt.show()

**Exercise D.** Store the correlation between the scores of the reward model trained on your annotator and the gold reward, `rho_taste`, in `taste_rho`.

In [ ]:
taste_rho = None   # your answer

check("Exercise D", taste_rho, rho_taste)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
taste_rho = rho_taste

check("Exercise D", taste_rho, rho_taste)

## 9. Going further (optional): checking the gradients

*Lecture: [Going further (optional)](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-04/lecture.html#going-further-optional)*

**From the formulas to the code.** `loss` is the weighted loss $L_w$ for the first four sentences of the corpus with the weights `w0`, and `g0` is its gradient from the backward pass `grad_weighted`. For one entry of each of four tables, the loop adds and subtracts $h = 10^{-6}$, and `num = (up - down) / 2e-6` is the finite difference of the lecture. `worst` is the largest relative difference between `num` and the entry of `g0`.

In [ ]:
lm0 = NeuralLM(seed=0); ids0 = IDS[:4]; w0 = np.array([0.5, -1.0, 2.0, 0.3])
loss = lambda: -(w0 * lm0.seq_logprob(ids0)).sum()
g0 = lm0.grad_weighted(ids0, w0); worst = 0.0
for k, idx in (("E", (int(ids0[0, 2]), 3)), ("W1", (5, 7)), ("W2", (5, int(ids0[0, 2]))), ("pos", (2, 4))):
    lm0.P[k][idx] += 1e-6; up = loss(); lm0.P[k][idx] -= 2e-6; down = loss(); lm0.P[k][idx] += 1e-6
    num = (up - down) / 2e-6; worst = max(worst, abs(num - g0[k][idx]) / (abs(num) + abs(g0[k][idx]) + 1e-12))
    print(f"{k:4s} backward pass {g0[k][idx]:+.6e}   finite difference {num:+.6e}")
print(f"largest relative difference: {worst:.1e}")

## Application challenges

Each challenge transfers the ideas of the day to a field of study. Choose the one closest to your programme and solve it in a copy of this notebook.

| Field | Challenge |
|---|---|
| Computer Science and Engineering | Give the reward model the count of repeated words as an extra feature, retrain it and test whether the hack of section 7 disappears. |
| Electronics and Communication Engineering | Treat the annotator as a noisy channel and estimate its error rate from the held-out accuracy of a reward model trained on clean labels. |
| Electrical and Electronics Engineering | Plot the KL against the iteration for several values of beta and relate the plateau to the closed-form optimum. |
| Mechanical Engineering | Design a gold reward for maintenance reports with three properties and a reward model that can see only two of them; predict the hack before running RLHF. |
| Biomedical Engineering | Discuss which blind spot of a learned reward would be most dangerous for a model that writes patient instructions, and how you would test for it. |
| Civil Engineering | Repeat the sweep of section 7 with half the preference pairs and report whether the best beta changes. |
| Aeronautical Engineering | Stop the beta = 0 run at the iteration with the highest gold reward and argue why early stopping is not a substitute for the KL penalty. |
| Biotechnology | Change the gold reward so that it rewards one extra phrase and measure how quickly RLHF finds it. |

## After the notebook

The self-assessment and the reflection are in the [interactive lab](https://utkukose.github.io/rl-alignment-veltech-NB-lecture/days/day-04/lab.html), tabs *Check yourself* and *Reflect and export*. The daily task and the research assignment are on the [day overview](https://github.com/utkukose/rl-alignment-veltech-NB-lecture/tree/main/days/day-04).

---

<div align="center">

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

</div>